In [ ]:
# Uncomment and run this cell if you're on Colab or Kaggle
# !git clone https://github.com/juancgar/nlp-with-transformers-2026.git
# %cd nlp-with-transformers-2026
# from install import *
# install_requirements()

In [ ]:
#hide
from utils import *
setup_chapter()

In [ ]:
#hide
%env TOKENIZERS_PARALLELISM=false
# Opt out of Haystack's anonymous usage telemetry (must be set before importing haystack)
%env HAYSTACK_TELEMETRY_ENABLED=False

In [ ]:
#hide
# Suppress Haystack logging
import logging
for module in ["haystack", "haystack_integrations", "sentence_transformers"]:
    module_logger = logging.getLogger(module)
    module_logger.setLevel(logging.ERROR)

# Question Answering

<img alt="Marie Curie" width="500" caption="A Google search query and corresponding answer snippet" src="images/chapter07_marie-curie.png" id="marie-curie"/>

## Building a Review-Based QA System

### The Dataset

<img alt="Phone with Query" width="400" caption="A question about a product and the corresponding review (the answer span is underlined)" src="images/chapter07_phone.png" id="phone"/>

In [ ]:
from huggingface_hub import HfApi, RepoFolder

# SubjQA's loading script is no longer supported by Datasets; its Parquet
# conversion on the Hub stores each domain in its own folder
domains = [f.path for f in HfApi().list_repo_tree(
    "megagonlabs/subjqa", repo_type="dataset", revision="refs/convert/parquet")
    if isinstance(f, RepoFolder)]
domains

In [ ]:
#hide_output
from datasets import load_dataset

subjqa = load_dataset("megagonlabs/subjqa", data_dir="electronics",
                      revision="refs/convert/parquet")

In [ ]:
print(subjqa["train"]["answers"][1])

In [ ]:
import pandas as pd

dfs = {split: dset.to_pandas() for split, dset in subjqa.flatten().items()}

for split, df in dfs.items():
    print(f"Number of questions in {split}: {df['id'].nunique()}")

In [ ]:
#hide_output
qa_cols = ["title", "question", "answers.text", 
           "answers.answer_start", "context"]
sample_df = dfs["train"][qa_cols].sample(2, random_state=7)
sample_df

In [ ]:
start_idx = sample_df["answers.answer_start"].iloc[0][0]
end_idx = start_idx + len(sample_df["answers.text"].iloc[0][0])
sample_df["context"].iloc[0][start_idx:end_idx]

In [ ]:
import matplotlib.pyplot as plt

counts = {}
question_types = ["What", "How", "Is", "Does", "Do", "Was", "Where", "Why"]

for q in question_types:
    counts[q] = dfs["train"]["question"].str.startswith(q).value_counts()[True]

pd.Series(counts).sort_values().plot.barh()
plt.title("Frequency of Question Types")
plt.show()

In [ ]:
for question_type in ["How", "What", "Is"]:
    for question in (
        dfs["train"][dfs["train"].question.str.startswith(question_type)]
        .sample(n=3, random_state=42)['question']):
        print(question)

### Sidebar: The Stanford Question Answering Dataset

<img alt="SQuAD SotA" width="600" caption="Progress on the SQuAD 2.0 benchmark (image from Papers with Code)" src="images/chapter07_squad-sota.png" id="squad-sota"/>

### End sidebar

### Extracting Answers from Text

#### Span classification

<img alt="QA Head" caption="The span classification head for QA tasks" src="images/chapter07_qa-head.png" id="qa-head"/>

<img alt="SQuAD models" width="600" caption="A selection of extractive QA models on the Hugging Face Hub" src="images/chapter07_squad-models.png" id="squad-models"/> 

#### Tokenizing text for QA

In [ ]:
#hide_output
from transformers import AutoTokenizer

model_ckpt = "deepset/minilm-uncased-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
question = "How much music can this hold?"
context = """An MP3 is about 1 MB/minute, so about 6000 hours depending on \
file size."""
inputs = tokenizer(question, context, return_tensors="pt")

In [ ]:
#hide_input
input_df = pd.DataFrame.from_dict(tokenizer(question, context), orient="index")
input_df

In [ ]:
print(tokenizer.decode(inputs["input_ids"][0]))

In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(model_ckpt)

with torch.no_grad():
    outputs = model(**inputs)
print(outputs)

In [ ]:
start_logits = outputs.start_logits
end_logits = outputs.end_logits

In [ ]:
print(f"Input IDs shape: {inputs.input_ids.size()}")
print(f"Start logits shape: {start_logits.size()}")
print(f"End logits shape: {end_logits.size()}")

In [ ]:
#hide_input
#id qa-scores
#caption Predicted logits for the start and end tokens; the token with the highest score is colored in orange

# The idea for this visualisation comes from https://mccormickml.com/2020/03/10/question-answering-with-a-fine-tuned-BERT
import numpy as np
import matplotlib.pyplot as plt

s_scores = start_logits.detach().numpy().flatten()
e_scores = end_logits.detach().numpy().flatten()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
# Plot against token positions (repeated tokens like [SEP] would be merged otherwise)
token_ids = range(len(tokens))

fig, (ax1, ax2) = plt.subplots(nrows=2, sharex=True)
colors = ["C0" if s != np.max(s_scores) else "C1" for s in s_scores]
ax1.bar(x=token_ids, height=s_scores, color=colors)
ax1.set_ylabel("Start Scores")
colors = ["C0" if s != np.max(e_scores) else "C1" for s in e_scores]
ax2.bar(x=token_ids, height=e_scores, color=colors)
ax2.set_ylabel("End Scores")
plt.xticks(token_ids, tokens, rotation="vertical")
plt.show()

In [ ]:
import torch 

start_idx = torch.argmax(start_logits)  
end_idx = torch.argmax(end_logits) + 1  
answer_span = inputs["input_ids"][0][start_idx:end_idx]
answer = tokenizer.decode(answer_span)
print(f"Question: {question}")
print(f"Answer: {answer}")

> **Note (updated for Transformers v5):** the `question-answering` pipeline was removed in Transformers v5, so `pipe()` below re-implements it for a short context: softmax over the context tokens (plus `[CLS]`), score all spans of at most `max_answer_len` tokens, return the `top_k` best; with `handle_impossible_answer=True` the "no answer" span (start and end on `[CLS]`) competes too.

In [ ]:
def pipe(question, context, top_k=1, handle_impossible_answer=False,
         max_answer_len=15):
    inputs = tokenizer(question, context, return_tensors="pt",
                       return_offsets_mapping=True)
    offsets = inputs.pop("offset_mapping")[0].tolist()
    with torch.no_grad():
        outputs = model(**inputs)
    # Only context tokens and [CLS] (= "no answer") can be predicted
    allowed = torch.tensor([s == 1 for s in inputs.sequence_ids(0)])
    allowed[0] = True
    start_probs = outputs.start_logits[0].masked_fill(~allowed, -1e4).softmax(-1)
    end_probs = outputs.end_logits[0].masked_fill(~allowed, -1e4).softmax(-1)
    # Score of the "no answer" prediction: start and end on [CLS]
    null_score = (start_probs[0] * end_probs[0]).item()
    start_probs[0] = end_probs[0] = 0.0
    # Score all spans with start <= end < start + max_answer_len
    span_scores = torch.triu(start_probs[:, None] * end_probs[None, :])
    span_scores = torch.tril(span_scores, diagonal=max_answer_len - 1)
    top_scores, top_idxs = span_scores.flatten().topk(top_k)
    answers = []
    for score, idx in zip(top_scores.tolist(), top_idxs.tolist()):
        start_idx, end_idx = divmod(idx, span_scores.size(1))
        start, end = offsets[start_idx][0], offsets[end_idx][1]
        answers.append({"score": score, "start": start, "end": end,
                        "answer": context[start:end]})
    if handle_impossible_answer:
        answers.append({"score": null_score, "start": 0, "end": 0, "answer": ""})
        answers = sorted(answers, key=lambda x: x["score"], reverse=True)[:top_k]
    return answers if top_k > 1 else answers[0]

pipe(question=question, context=context, top_k=3)

In [ ]:
pipe(question="Why is there no data?", context=context, 
     handle_impossible_answer=True)

#### Dealing with long passages

In [ ]:
#hide_input
#id subjqa-dist
#caption Distribution of tokens for each question-context pair in the SubjQA training set
def compute_input_length(row):
    inputs = tokenizer(row["question"], row["context"])
    return len(inputs["input_ids"])

dfs["train"]["n_tokens"] = dfs["train"].apply(compute_input_length, axis=1)

fig, ax = plt.subplots()
dfs["train"]["n_tokens"].hist(bins=100, grid=False, ec="C0", ax=ax)
plt.xlabel("Number of tokens in question-context pair")
ax.axvline(x=512, ymin=0, ymax=1, linestyle="--", color="C1", 
           label="Maximum sequence length")
plt.legend()
plt.ylabel("Count")
plt.show()

<img alt="Sliding window" caption="How the sliding window creates multiple question-context pairs for long documents—the first bar corresponds to the question, while the second bar is the context captured in each window" src="images/chapter07_sliding-window.png" id="sliding-window"/>

> **Note (tokenizers 0.23):** Tokenizers ≥ 0.23.1 has a regression ([huggingface/tokenizers#2091](https://github.com/huggingface/tokenizers/issues/2091)): with `return_overflowing_tokens=True` only the first `max_length` tokens are split into windows and the rest of the context is dropped. That's why `requirements.txt` pins `tokenizers<0.23.1` (and therefore `transformers` 5.15). If the windows below come out shorter than in the book, check `tokenizers.__version__`.

In [ ]:
example = dfs["train"].iloc[0][["question", "context"]]
tokenized_example = tokenizer(example["question"], example["context"], 
                              return_overflowing_tokens=True, max_length=100, 
                              stride=25)

In [ ]:
for idx, window in enumerate(tokenized_example["input_ids"]):
    print(f"Window #{idx} has {len(window)} tokens")

In [ ]:
for window in tokenized_example["input_ids"]:
    print(f"{tokenizer.decode(window)} \n")

### Using Haystack to Build a QA Pipeline

<img alt="QA Architecture" caption="The retriever-reader architecture for modern QA systems" src="images/chapter07_retriever-reader.png" id="retriever-reader"/>

#### Initializing a document store

> **Note (updated for Haystack 3):** the book downloads and launches an Elasticsearch server here for Haystack 0.9's `ElasticsearchDocumentStore`. Haystack 3 (`haystack-ai`) has a new API, and for a few thousand reviews its `InMemoryDocumentStore` (BM25 and embedding retrieval, no server needed) is enough, so the Elasticsearch setup cells were removed.

In [ ]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

# Return the document embedding for later use with dense retriever
document_store = InMemoryDocumentStore(return_embedding=True)

In [ ]:
#hide
# It's a good idea to flush the document store with each notebook restart
if document_store.count_documents() > 0:
    document_store.delete_all_documents()

> **Note (updated for Haystack 3):** documents are now `Document` objects with a `content` field (formerly `text`) and a `meta` dict; there are no named indices, and labels are no longer stored in the document store.

In [ ]:
from haystack import Document

for split, df in dfs.items():
    # Exclude duplicate reviews
    docs = [Document(content=row["context"],
                     meta={"item_id": row["title"], "question_id": row["id"],
                           "split": split})
        for _,row in df.drop_duplicates(subset="context").iterrows()]
    document_store.write_documents(docs)

print(f"Loaded {document_store.count_documents()} documents")

#### Initializing a retriever

> **Note (updated for Haystack 3):** `ElasticsearchRetriever` becomes `InMemoryBM25Retriever` (BM25L by default, so scores and rankings differ slightly from Elasticsearch). Components are called with `run()` and return a dict, and filters now use an explicit `field`/`operator`/`value` syntax, which our `item_filter()` helper builds.

In [ ]:
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

# We keep the book's variable name, although BM25 now runs in memory
es_retriever = InMemoryBM25Retriever(document_store=document_store)

In [ ]:
def item_filter(item_id, split=None):
    """Haystack 3 version of the filter {"item_id": [item_id], "split": [split]}"""
    conditions = [{"field": "meta.item_id", "operator": "==", "value": item_id}]
    if split is not None:
        conditions.append({"field": "meta.split", "operator": "==", "value": split})
    return {"operator": "AND", "conditions": conditions}

item_id = "B0074BW614"
query = "Is it good for reading?"
retrieved_docs = es_retriever.run(
    query=query, top_k=3, filters=item_filter(item_id, "train"))["documents"]

In [ ]:
print(retrieved_docs[0])

#### Initializing a reader

> **Note (updated for Haystack 3):** `FARMReader` is replaced by `TransformersExtractiveReader` (from `transformers-haystack`), which uses the same sliding windows (`max_seq_length`, `stride`). With `no_answer=True` it adds a "no answer" (`data=None`). Its scores are not softmax probabilities: each span gets an independent calibrated score, and the no-answer score is the probability that none of the returned spans is correct.

In [ ]:
#hide_output
from haystack_integrations.components.readers.transformers import TransformersExtractiveReader

model_ckpt = "deepset/minilm-uncased-squad2"
max_seq_length, doc_stride = 384, 128
reader = TransformersExtractiveReader(model=model_ckpt,
                                      max_seq_length=max_seq_length,
                                      stride=doc_stride, no_answer=True)
reader.warm_up()  # loads the model and tokenizer

In [ ]:
print(reader.run(query=question, documents=[Document(content=context)], top_k=1))

#### Putting it all together

> **Note (updated for Haystack 3):** `ExtractiveQAPipeline` is gone; we add both components to a `Pipeline`, connect the retriever's `documents` output to the reader, and pass each component's inputs under its name. Answers no longer have a `context` field, so `get_snippet()` cuts it from the document.

In [ ]:
from haystack import Pipeline

pipe = Pipeline()
pipe.add_component("retriever", es_retriever)
pipe.add_component("reader", reader)
pipe.connect("retriever.documents", "reader.documents")

In [ ]:
def get_snippet(answer, window=75):
    """Text around the answer span (the `context` of Haystack 0.9 answers)"""
    if answer.document is None:  # the "no answer" prediction
        return ""
    start, end = answer.document_offset.start, answer.document_offset.end
    return answer.document.content[max(0, start - window):end + window]

n_answers = 3
preds = pipe.run({"retriever": {"query": query, "top_k": 3,
                                "filters": item_filter(item_id, "train")},
                  "reader": {"query": query, "top_k": n_answers}})
answers = preds["reader"]["answers"]

print(f"Question: {query} \n")
for idx in range(n_answers):
    print(f"Answer {idx+1}: {answers[idx].data}")
    print(f"Review snippet: ...{get_snippet(answers[idx])}...")
    print("\n\n")

## Improving Our QA Pipeline

### Evaluating the Retriever

> **Note (updated for Haystack 3):** there are no `EvalDocuments`/`EvalAnswers` nodes anymore; you run the retriever and pass its results plus the ground truth to evaluators such as `DocumentRecallEvaluator`, as our `EvalRetrieverPipeline` does. As in the book, a document is relevant if it contains an answer, but unlike Haystack 0.9 (which counted unanswerable questions as hits) we only evaluate answerable questions, so recall is lower than in the book.

In [ ]:
from haystack.components.evaluators import DocumentRecallEvaluator

class EvalRetrieverPipeline:
    def __init__(self, retriever):
        self.retriever = retriever
        self.eval_retriever = DocumentRecallEvaluator(mode="single_hit")

    def retrieve(self, query, top_k, filters):
        if isinstance(self.retriever, Pipeline):  # dense retrieval (see DPR below)
            result = self.retriever.run({"text_embedder": {"text": query},
                "retriever": {"top_k": top_k, "filters": filters}})
            return result["retriever"]["documents"]
        return self.retriever.run(query=query, top_k=top_k,
                                  filters=filters)["documents"]

    def recall(self, relevant_docs, retrieved_docs, top_k):
        # Only the top-k documents of each ranking count
        return self.eval_retriever.run(ground_truth_documents=relevant_docs,
            retrieved_documents=[docs[:top_k] for docs in retrieved_docs])["score"]


pipe = EvalRetrieverPipeline(es_retriever)

> **Note (updated for Haystack 3):** without Haystack's `Label` class, each label is a dict with the question, all its answers (empty for unanswerable questions), the review, and metadata. This also avoids a problem in the book's code, where labels of the same question shared an `id`, so only one answer per question was stored.

In [ ]:
labels = []
for i, row in dfs["test"].iterrows():
    # Metadata used for filtering in the Retriever
    meta = {"item_id": row["title"], "question_id": row["id"]}
    # Questions without answers get an empty list of answers
    labels.append({"question": row["question"],
                   "answers": list(row["answers.text"]),
                   "context": row["context"], "meta": meta})

In [ ]:
print(labels[0])

In [ ]:
print(f"Loaded {len(labels)} question-answer pairs")

In [ ]:
# Merge the labels of identical questions about the same product, like
# Haystack 0.9's get_all_labels_aggregated(open_domain=True,
# aggregate_by_meta=["item_id"])
labels_agg = {}
for l in labels:
    meta = {"item_id": l["meta"]["item_id"]}
    agg = labels_agg.setdefault((l["question"], meta["item_id"]), {
        "question": l["question"], "answers": [], "contexts": [], "meta": meta})
    agg["answers"] += [a for a in l["answers"] if a not in agg["answers"]]
    if l["context"] not in agg["contexts"]:
        agg["contexts"].append(l["context"])
labels_agg = list(labels_agg.values())
print(len(labels_agg))

In [ ]:
print(labels_agg[109])

In [ ]:
def run_pipeline(pipeline, top_k_retriever=10):
    relevant_docs, retrieved_docs = [], []
    for l in labels_agg:
        # Questions without answers have no relevant documents
        if not l["answers"]:
            continue
        filters = item_filter(l["meta"]["item_id"], "test")
        retrieved_docs.append(
            pipeline.retrieve(l["question"], top_k_retriever, filters))
        # A document is relevant if it contains one of the answers
        relevant_docs.append([doc for doc in document_store.filter_documents(filters)
            if any(a.lower() in doc.content.lower() for a in l["answers"])])
    return relevant_docs, retrieved_docs

In [ ]:
relevant_docs, retrieved_docs = run_pipeline(pipe, top_k_retriever=3)
print(f"Recall@3: {pipe.recall(relevant_docs, retrieved_docs, top_k=3):.2f}")

In [ ]:
#hide_output
def evaluate_retriever(retriever, topk_values = [1,3,5,10,20]):
    topk_results = {}
    # Create Pipeline
    p = EvalRetrieverPipeline(retriever)
    # Retrieve the top documents for each question-answers pair in test set
    # once with the largest k; the rankings for smaller k are its prefixes
    relevant_docs, retrieved_docs = run_pipeline(p, top_k_retriever=max(topk_values))

    for topk in topk_values:
        # Get metrics
        topk_results[topk] = {"recall": p.recall(relevant_docs, retrieved_docs, topk)}

    return pd.DataFrame.from_dict(topk_results, orient="index")


es_topk_df = evaluate_retriever(es_retriever)

In [ ]:
def plot_retriever_eval(dfs, retriever_names):
    fig, ax = plt.subplots()
    for df, retriever_name in zip(dfs, retriever_names):
        df.plot(y="recall", ax=ax, label=retriever_name)
    plt.xticks(df.index)
    plt.ylabel("Top-k Recall")
    plt.xlabel("k")
    plt.show()
    
plot_retriever_eval([es_topk_df], ["BM25"])

#### Dense Passage Retrieval

<img alt="DPR Architecture" caption="DPR's bi-encoder architecture for computing the relevance of a document and query" src="images/chapter07_dpr.png" id="dpr"/>

> **Note (updated for Haystack 3):** `DensePassageRetriever` is split into Sentence Transformers embedders (from `sentence-transformers-haystack`) with the Sentence Transformers versions of the same DPR checkpoints: the query embedder plus an `InMemoryEmbeddingRetriever` (dot product) form the `dpr_retriever` pipeline, and instead of `update_embeddings()` a document embedder embeds all reviews, which we write back to the store.

In [ ]:
#hide_output
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder)

dpr_retriever = Pipeline()
dpr_retriever.add_component("text_embedder", SentenceTransformersTextEmbedder(
    model="sentence-transformers/facebook-dpr-question_encoder-single-nq-base",
    progress_bar=False))
# Return the document embeddings as well (needed for RAG at the end of the chapter)
dpr_retriever.add_component("retriever", InMemoryEmbeddingRetriever(
    document_store=document_store, return_embedding=True))
dpr_retriever.connect("text_embedder.embedding", "retriever.query_embedding")
dpr_retriever.warm_up()  # loads the question encoder

In [ ]:
#hide_output
from haystack.document_stores.types import DuplicatePolicy

doc_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/facebook-dpr-ctx_encoder-single-nq-base")
embedded_docs = doc_embedder.run(document_store.filter_documents())["documents"]
document_store.write_documents(embedded_docs, policy=DuplicatePolicy.OVERWRITE)

In [ ]:
dpr_topk_df = evaluate_retriever(dpr_retriever)
plot_retriever_eval([es_topk_df, dpr_topk_df], ["BM25", "DPR"])

### Evaluating the Reader

> **Note (updated for Haystack 3):** FARM's `compute_exact()` and `compute_f1()` are gone; below are the same functions from the official SQuAD 2.0 evaluation script.

In [ ]:
import collections
import re
import string

def normalize_answer(s):
    """Lowercase and remove punctuation, articles, and extra whitespace"""
    s = "".join(ch for ch in s.lower() if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def compute_exact(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def compute_f1(a_gold, a_pred):
    gold_toks = normalize_answer(a_gold).split()
    pred_toks = normalize_answer(a_pred).split()
    common = collections.Counter(gold_toks) & collections.Counter(pred_toks)
    num_same = sum(common.values())
    # If either is "no answer", F1 is 1 if they agree and 0 otherwise
    if len(gold_toks) == 0 or len(pred_toks) == 0:
        return int(gold_toks == pred_toks)
    if num_same == 0:
        return 0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return (2 * precision * recall) / (precision + recall)

In [ ]:
pred = "about 6000 hours"
label = "6000 hours"
print(f"EM: {compute_exact(label, pred)}")
print(f"F1: {compute_f1(label, pred)}")

In [ ]:
pred = "about 6000 dollars"
print(f"EM: {compute_exact(label, pred)}")
print(f"F1: {compute_f1(label, pred)}")

> **Note (updated for Haystack 3):** instead of an `EvalAnswers` node, we run the reader on the review(s) each test question was asked about and compute top-1 EM and F1 against the best-matching gold answer (`AnswerExactMatchEvaluator` only does exact string matches against a single answer). As in the book, unanswerable questions and "no answer" predictions are left out.

In [ ]:
#hide_output
def top_1_scores(preds, golds):
    """Mean EM and F1 of the top-1 predictions (best match among gold answers)"""
    return {"top_1_em": np.mean([max(compute_exact(g, p) for g in gs)
                                 for p, gs in zip(preds, golds)]),
            "top_1_f1": np.mean([max(compute_f1(g, p) for g in gs)
                                 for p, gs in zip(preds, golds)])}

def evaluate_reader(reader):
    preds, golds = [], []
    for l in labels_agg:
        # Like Haystack 0.9, only score the questions that have an answer
        if not l["answers"]:
            continue
        # Give the reader the review(s) the question was asked about
        docs = [Document(content=context) for context in l["contexts"]]
        answers = reader.run(query=l["question"], documents=docs, top_k=1,
                             no_answer=False)["answers"]
        preds.append(answers[0].data if answers else "")
        golds.append(l["answers"])
    return top_1_scores(preds, golds)

reader_eval = {}
reader_eval["Fine-tune on SQuAD"] = evaluate_reader(reader)

In [ ]:
def plot_reader_eval(reader_eval):
    fig, ax = plt.subplots()
    df = pd.DataFrame.from_dict(reader_eval)
    df.plot(kind="bar", ylabel="Score", rot=0, ax=ax)
    ax.set_xticklabels(["EM", "F1"])
    plt.legend(loc='upper left')
    plt.show()

plot_reader_eval(reader_eval)

### Domain Adaptation

<img alt="SQuAD Schema" caption="Visualization of the SQuAD JSON format" src="images/chapter07_squad-schema.png" id="squad-schema"/>

In [ ]:
def create_paragraphs(df):
    paragraphs = []
    id2context = dict(zip(df["review_id"], df["context"]))
    for review_id, review in id2context.items():
        qas = []
        # Filter for all question-answer pairs about a specific context
        review_df = df.query(f"review_id == '{review_id}'")
        id2question = dict(zip(review_df["id"], review_df["question"]))
        # Build up the qas array
        for qid, question in id2question.items():
            # Filter for a single question ID
            question_df = df.query(f"id == '{qid}'").to_dict(orient="list")
            ans_start_idxs = question_df["answers.answer_start"][0].tolist()
            ans_text = question_df["answers.text"][0].tolist()
            # Fill answerable questions
            if len(ans_start_idxs):
                answers = [
                    {"text": text, "answer_start": answer_start}
                    for text, answer_start in zip(ans_text, ans_start_idxs)]
                is_impossible = False
            else:
                answers = []
                is_impossible = True
            # Add question-answer pairs to qas
            qas.append({"question": question, "id": qid, 
                        "is_impossible": is_impossible, "answers": answers})
        # Add context and question-answer pairs to paragraphs
        paragraphs.append({"qas": qas, "context": review})
    return paragraphs

In [ ]:
#hide_output
product = dfs["train"].query("title == 'B00001P4ZH'")
create_paragraphs(product)

```python
[{'qas': [{'question': 'How is the bass?',
    'id': '2543d296da9766d8d17d040ecc781699',
    'is_impossible': True,
    'answers': []}],
  'context': 'I have had Koss headphones ...',
    'id': 'd476830bf9282e2b9033e2bb44bbb995',
    'is_impossible': False,
    'answers': [{'text': 'Bass is weak as expected', 'answer_start': 1302},
     {'text': 'Bass is weak as expected, even with EQ adjusted up',
      'answer_start': 1302}]}],
  'context': 'To anyone who hasn\'t tried all ...'},
 {'qas': [{'question': 'How is the bass?',
    'id': '455575557886d6dfeea5aa19577e5de4',
    'is_impossible': False,
    'answers': [{'text': 'The only fault in the sound is the bass',
      'answer_start': 650}]}],
  'context': "I have had many sub-$100 headphones ..."}]
```

In [ ]:
import json

def convert_to_squad(dfs):
    for split, df in dfs.items():
        subjqa_data = {}
        # Create `paragraphs` for each product ID
        groups = (df.groupby("title").apply(create_paragraphs)
            .to_frame(name="paragraphs").reset_index())
        subjqa_data["data"] = groups.to_dict(orient="records")
        # Save the result to disk
        with open(f"electronics-{split}.json", "w+", encoding="utf-8") as f:
            json.dump(subjqa_data, f)
            
convert_to_squad(dfs)

> **Note (updated for Haystack 3):** `FARMReader.train()` is gone, so we fine-tune with the Transformers `Trainer` and the standard extractive QA recipe: split each question-review pair into windows and use the offset mapping to turn the answer's character span into start/end token positions (`[CLS]` for no answer). We tokenize the SubjQA `Dataset` directly (it has SQuAD's fields) instead of the JSON files, then load the saved model into a new `TransformersExtractiveReader`.

In [ ]:
def prepare_train_features(examples, tokenizer):
    # Split long reviews into overlapping windows, as the reader does
    inputs = tokenizer(examples["question"], examples["context"],
                       truncation="only_second", max_length=max_seq_length,
                       stride=doc_stride, return_overflowing_tokens=True,
                       return_offsets_mapping=True, padding="max_length")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")
    inputs["start_positions"], inputs["end_positions"] = [], []
    for i, offsets in enumerate(offset_mapping):
        answers = examples["answers"][sample_map[i]]
        # First and last token of the review in this window
        sequence_ids = inputs.sequence_ids(i)
        ctx_start = sequence_ids.index(1)
        ctx_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)
        # No answer (in this window): start and end point to [CLS]
        start_pos = end_pos = 0
        if len(answers["answer_start"]) > 0:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])
            # Map the answer's characters to tokens if it's inside the window
            if (offsets[ctx_start][0] <= start_char
                    and offsets[ctx_end][1] >= end_char):
                start_pos, end_pos = ctx_start, ctx_end
                while start_pos <= ctx_end and offsets[start_pos][0] <= start_char:
                    start_pos += 1
                start_pos -= 1
                while end_pos >= ctx_start and offsets[end_pos][1] >= end_char:
                    end_pos -= 1
                end_pos += 1
        inputs["start_positions"].append(start_pos)
        inputs["end_positions"].append(end_pos)
    return inputs

In [ ]:
#hide_output
from transformers import Trainer, TrainingArguments

def train_reader(model_ckpt, output_dir, n_epochs=1, batch_size=16):
    qa_tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
    qa_model = AutoModelForQuestionAnswering.from_pretrained(model_ckpt)
    train_ds, dev_ds = [subjqa[split].map(
        prepare_train_features, batched=True, fn_kwargs={"tokenizer": qa_tokenizer},
        remove_columns=subjqa[split].column_names)
        for split in ["train", "validation"]]
    # Learning rate and warmup (20% of the steps) are FARMReader.train()'s defaults
    training_args = TrainingArguments(
        output_dir=output_dir, num_train_epochs=n_epochs, learning_rate=1e-5,
        warmup_steps=0.2, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size, eval_strategy="epoch",
        save_strategy="no", log_level="error", disable_tqdm=False,
        push_to_hub=False)
    trainer = Trainer(model=qa_model, args=training_args, train_dataset=train_ds,
                      eval_dataset=dev_ds, processing_class=qa_tokenizer)
    trainer.train()
    trainer.save_model()  # model and tokenizer go to output_dir
    return output_dir

reader = TransformersExtractiveReader(
    model=train_reader(model_ckpt, "minilm-uncased-squad2-subjqa"),
    max_seq_length=max_seq_length, stride=doc_stride, no_answer=True)

In [ ]:
reader_eval["Fine-tune on SQuAD + SubjQA"] = evaluate_reader(reader)
plot_reader_eval(reader_eval)

In [ ]:
# MiniLM without any QA fine-tuning: train_reader() adds a new, randomly
# initialized QA head when it loads this checkpoint
minilm_ckpt = "microsoft/MiniLM-L12-H384-uncased"

In [ ]:
#hide_output
minilm_reader = TransformersExtractiveReader(
    model=train_reader(minilm_ckpt, "minilm-uncased-subjqa"),
    max_seq_length=max_seq_length, stride=doc_stride, no_answer=True)

In [ ]:
reader_eval["Fine-tune on SubjQA"] = evaluate_reader(minilm_reader)
plot_reader_eval(reader_eval)

### Evaluating the Whole QA Pipeline

> **Note (updated for Haystack 3):** we run the fine-tuned reader behind a BM25 retriever (top 10 reviews, top 4 answers as in the book) and score its best answer for each answerable question. Unlike Haystack 0.9's `EvalAnswers`, questions where retrieval failed are not skipped.

In [ ]:
#hide_output
# Initialize retriever-reader pipeline (a component can only belong to one
# pipeline, so we create a new BM25 retriever)
pipe = Pipeline()
pipe.add_component("retriever", InMemoryBM25Retriever(document_store=document_store))
pipe.add_component("reader", reader)
pipe.connect("retriever.documents", "reader.documents")
# Evaluate!
preds, golds = [], []
for l in labels_agg:
    if not l["answers"]:
        continue
    result = pipe.run({
        "retriever": {"query": l["question"], "top_k": 10,
                      "filters": item_filter(l["meta"]["item_id"], "test")},
        "reader": {"query": l["question"], "top_k": 4, "no_answer": False}})
    answers = result["reader"]["answers"]
    preds.append(answers[0].data if answers else "")
    golds.append(l["answers"])
# Extract metrics from reader
reader_eval["QA Pipeline (top-1)"] = top_1_scores(preds, golds)

In [ ]:
#hide_input
#id reader-vs-pipeline
#caption Comparison of EM and _F_~1~ scores for the reader against the whole QA pipeline
plot_reader_eval({"Reader": reader_eval["Fine-tune on SQuAD + SubjQA"], 
                  "QA pipeline (top-1)": reader_eval["QA Pipeline (top-1)"]})

## Going Beyond Extractive QA

<img alt="RAG Architecture" width="600" caption="The RAG architecture for fine-tuning a retriever and generator end-to-end (courtesy of Ethan Perez)" src="images/chapter07_rag-architecture.png" id="rag"/>

> **Note (updated for Haystack 3):** Haystack 3 has no `RAGenerator`/`GenerativeQAPipeline` (its generators prompt LLMs), so we call the book's `RagTokenForGeneration` model from Transformers directly, as `RAGenerator` did internally: RAG's question encoder scores the reviews retrieved by DPR, and the BART generator marginalizes over them while decoding (RAG's own Wikipedia `RagRetriever` isn't needed). We preferred this to a generic retrieve-then-generate setup with a seq2seq model because it keeps the book's model and end-to-end RAG decoding.

In [ ]:
#hide_output
from transformers import RagTokenizer, RagTokenForGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rag_ckpt = "facebook/rag-token-nq"
rag_tokenizer = RagTokenizer.from_pretrained(rag_ckpt)
generator = RagTokenForGeneration.from_pretrained(rag_ckpt).to(device).eval()

In [ ]:
def rag_generate(query, documents, top_k=3, num_beams=5):
    config = generator.config
    # Score the documents with RAG's question encoder and their DPR embeddings
    question_ids = rag_tokenizer(query, return_tensors="pt")["input_ids"].to(device)
    with torch.no_grad():
        query_embedding = generator.question_encoder(question_ids)[0]
    doc_embeddings = torch.tensor([doc.embedding for doc in documents],
                                  device=device)
    doc_scores = query_embedding @ doc_embeddings.T
    # Generator inputs have the form "<title> / <document> // <question>"
    # (we don't use titles, as in the book)
    prefix = getattr(config.generator, "prefix", None) or ""
    rag_inputs = [(prefix + config.title_sep + doc.content + config.doc_sep
                   + query).replace("  ", " ") for doc in documents]
    context = rag_tokenizer.generator(
        rag_inputs, max_length=config.max_combined_length, padding="max_length",
        truncation=True, return_tensors="pt").to(device)
    generated_ids = generator.generate(
        context_input_ids=context["input_ids"],
        context_attention_mask=context["attention_mask"], doc_scores=doc_scores,
        n_docs=len(documents), num_beams=num_beams, num_return_sequences=top_k,
        max_length=200, min_length=2)
    return rag_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

In [ ]:
def generate_answers(query, top_k_generator=3):
    docs = dpr_retriever.run({"text_embedder": {"text": query},
                              "retriever": {"top_k": 5,
                                            "filters": item_filter("B0074BW614")}})
    answers = rag_generate(query, docs["retriever"]["documents"],
                           top_k=top_k_generator)
    print(f"Question: {query} \n")
    for idx in range(top_k_generator):
        print(f"Answer {idx+1}: {answers[idx]}")

In [ ]:
generate_answers(query)

In [ ]:
generate_answers("What is the main drawback?")

## Conclusion

<img alt="QA Pyramid" caption="The QA hierarchy of needs" src="images/chapter07_qa-pyramid.png" id="qa-pyramid"/> 